# p02 · The ordered address, and what 'hyperindex' refers to
The Hyperwebster, as presented in Vsauce's *The Banach–Tarski Paradox* (transcript, `sites.google.com/site/vsaucetranscripts/scripts/the-banach-tarski-paradox`): "a dictionary … [that] lists every single possible word of any length formed from the 26 letters in the English alphabet"; "if they put all the words that begin with 'a' in a volume titled 'A,' they wouldn't have to print the initial 'a'." Below: the dictionary as a computable address, the volume-A trick checked exhaustively, and the definition tested on both libraries of p01.

In [1]:
import importlib.util, os, sys, time, math, hashlib, json
from collections import Counter
PROJ = os.environ.get("PROJ", "/home/rendier/Projects/ThePlace")
NB = os.path.join(PROJ, "FourthAgePapers/HyperindexingSystem/notebooks")
CORPUS = open(os.path.join(NB, "corpus_sample.txt"), encoding="utf-8").read()   # fixed 4000-char sample, 97-symbol keyboard set
def load(rel, name="m"):
    """Load a source file from the repo by path, unmodified."""
    spec = importlib.util.spec_from_file_location(name, os.path.join(PROJ, rel))
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m
print("corpus:", len(CORPUS), "chars, sha256", hashlib.sha256(CORPUS.encode()).hexdigest()[:16])

corpus: 4000 chars, sha256 3105964b9180566a


In [2]:
# Listing 1 - the ordered address (bijective base-N): text <-> non-negative integer
def address(text, alphabet):
    """Rank of `text` among all strings over `alphabet`, shortest first, then alphabetical."""
    N, a = len(alphabet), 0
    for ch in text:
        a = a * N + alphabet.index(ch) + 1
    return a - 1

def text_at(a, alphabet):
    """Inverse of address(): the string whose rank is `a`."""
    N, out = len(alphabet), []
    a += 1
    while a > 0:
        a, r = divmod(a - 1, N)
        out.append(alphabet[r])
    return "".join(reversed(out))

def page_at(loc, alphabet, k):
    """Fixed-length page: the k-digit base-N numeral of `loc` (digit 0 = first symbol)."""
    N, out = len(alphabet), []
    for _ in range(k):
        loc, r = divmod(loc, N)
        out.append(alphabet[r])
    return "".join(reversed(out))

def loc_of(page, alphabet):
    """Inverse of page_at()."""
    N, loc = len(alphabet), 0
    for ch in page:
        loc = loc * N + alphabet.index(ch)
    return loc

# Listing 2 - an invertible scrambler of [0, N): LCG + xorshift, cycle-walked into range
def make_scrambler(N, a=1664525, c=1013904223, s1=7, s2=5):
    e = max(1, (N - 1).bit_length()); M = 1 << e; mask = M - 1
    a_inv = pow(a, -1, M)                       # a odd -> invertible mod 2^e
    def f(x):                                   # bijection on [0, M)
        x = (a * x + c) & mask                  # Hull-Dobell: a = 1 mod 4, c odd
        x ^= x >> s1
        x ^= (x << s2) & mask
        return x
    def finv(y):
        z = y
        for _ in range(e // s2 + 1): z = y ^ ((z << s2) & mask)
        y = z
        for _ in range(e // s1 + 1): z = y ^ (z >> s1)
        return (a_inv * (z - c)) & mask
    def scramble(x):                            # bijection on [0, N)
        x = f(x)
        while x >= N: x = f(x)
        return x
    def unscramble(y):
        y = finv(y)
        while y >= N: y = finv(y)
        return y
    return scramble, unscramble

print('listings defined')

listings defined


## The dictionary as an address (26 letters, shortest first)
Transcript, first rows. `address` and `text_at` are Listing 1.

In [3]:
A26 = "abcdefghijklmnopqrstuvwxyz"
print("address   word")
for n in list(range(0, 30)) + [700, 701, 702]:
    w = text_at(n, A26); assert address(w, A26) == n
    print("%-9d %r" % (n, w))

address   word
0         'a'
1         'b'
2         'c'
3         'd'
4         'e'
5         'f'
6         'g'
7         'h'
8         'i'
9         'j'
10        'k'
11        'l'
12        'm'
13        'n'
14        'o'
15        'p'
16        'q'
17        'r'
18        's'
19        't'
20        'u'
21        'v'
22        'w'
23        'x'
24        'y'
25        'z'
26        'aa'
27        'ab'
28        'ac'
29        'ad'
700       'zy'
701       'zz'
702       'aaa'


Address 0 is `a`, 25 is `z`, 26 is `aa`, 701 is `zz`, 702 is `aaa`. Every finite word has exactly one address and every non-negative integer is the address of exactly one word.

## Volume A: dropping the initial letter
The map `w ↦ 'a'+w` from all words (including the empty word) onto the words that begin with `a`, checked for **all** words of length ≤ 4 over 26 letters (475,255 words):

In [4]:
from itertools import product
words = [""] + ["".join(p) for n in range(1, 5) for p in product(A26, repeat=n)]
volA = {w for w in words if w[:1] == "a"}
stripped = {w[1:] for w in volA if len(w) <= 4}
print(len(words), "words of length <= 4 | volume A holds", len(volA), "| stripped of the initial letter:", len(stripped))
allshort = {w for w in words if len(w) <= 3}
assert {w[1:] for w in volA if len(w) <= 4} >= allshort
assert {"a" + w for w in allshort} <= volA
print("every word of length <= 3 is a volume-A word with its initial 'a' removed, and back: True")

475255 words of length <= 4 | volume A holds 18279 | stripped of the initial letter: 18279
every word of length <= 3 is a volume-A word with its initial 'a' removed, and back: True


**In the address.** Prepending `a` to a word of length ℓ adds exactly `N^ℓ` to its numeral value (digit value 1 at the new most significant place), so the volume-A word and the word it came from differ by a power of N. Exhaustive for all words up to length 4:

In [5]:
N = 26
ok = all(address("a" + w, A26) - address(w, A26) == N ** len(w) for w in words if len(w) <= 4)
print("address('a'+w) - address(w) == 26**len(w) for every word w with len(w) <= 4:", ok); assert ok

address('a'+w) - address(w) == 26**len(w) for every word w with len(w) <= 4: True


## What 'hyperindex' refers to, tested on both libraries
A **hyperindex** is a bijection between a set of texts and an initial segment of the non-negative integers such that (i) the integer is *computed from the text* and the text from the integer (no table is consulted or stored), and (ii) it is exact. Two properties then separate the members of the family, measured on the 24,389-page toy library of p01:

In [6]:
ALPHA = "abcdefghijklmnopqrstuvwxyz ,."; K = 3; NP = len(ALPHA) ** K
scramble, unscramble = make_scrambler(NP)
ordered_loc  = lambda page: loc_of(page, ALPHA)
babel_loc    = lambda page: unscramble(loc_of(page, ALPHA))
# (i)+(ii): exact bijection, no table
ex_o = len({ordered_loc(page_at(L, ALPHA, K)) for L in range(NP)}) == NP
ex_b = len({babel_loc(page_at(L, ALPHA, K)) for L in range(NP)}) == NP
# prefix structure: location('b'+w) - location('a'+w) should be the same for every w if the first symbol is a digit
ws = [a + b for a in ALPHA for b in ALPHA]
def spread(loc): return len({loc("b" + w) - loc("a" + w) for w in ws})
print("%-10s %-22s %-28s %s" % ("library", "exact bijection (N pages)", "distinct (loc('b'+w)-loc('a'+w))", "locations of 'aab','aac' differ by"))
print("%-10s %-22s %-28d %d" % ("ordered", ex_o, spread(ordered_loc), ordered_loc("aac") - ordered_loc("aab")))
print("%-10s %-22s %-28d %d" % ("scrambled", ex_b, spread(babel_loc), babel_loc("aac") - babel_loc("aab")))
assert ex_o and ex_b and spread(ordered_loc) == 1 and spread(babel_loc) > 100

library    exact bijection (N pages) distinct (loc('b'+w)-loc('a'+w)) locations of 'aab','aac' differ by
ordered    True                   1                            1
scrambled  True                   762                          -13427


Both are hyperindices. The ordered one keeps the structure of the text in the integer: a prefix is a place value, so the volume trick, neighbouring pages and one-symbol edits are arithmetic on the address (`+1`, `+N^ℓ`). The scrambled one destroys it on purpose: neighbouring pages are unrelated locations. The rest of this paper is the ordered family, in every form it took, and what each form kept and lost.